# Data Analysis
This notebook explores the data generated with the `scripts/build_datasets.py` script. That dataset is dependent on the batch of replays used to create it, so results may vary from run to run.

## Setup

In [3]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from starcraft_predictor.replays.unit_tracker import TRACKED_UNIT_TYPES, TRACKED_UPGRADE_TYPES
from starcraft_predictor.replays.player_stats_tracker import PLAYER_STATS_FIELDS

### Helper functions

In [4]:
def create_tabbed_plots(plot_functions, titles, overall_title=None):
    """
    Create a tabbed interface with multiple plots.
    
    Args:
        plot_functions (list): List of functions that create plots
        titles (list): List of titles for each tab
        overall_title (str, optional): Title for the entire tabbed interface
    """
    # Create tabs
    tab = widgets.Tab()
    tab.children = [widgets.Output() for _ in plot_functions]
    
    # Set tab titles
    for i, title in enumerate(titles):
        tab.set_title(i, title)
    
    # Create plots in each tab
    for i, plot_func in enumerate(plot_functions):
        with tab.children[i]:
            plot_func()
            plt.show()
    
    # Create container for title and tabs
    container = widgets.VBox()
    
    # Add title if provided
    if overall_title:
        title_widget = widgets.HTML(value=f"<h2>{overall_title}</h2>")
        container.children = [title_widget, tab]
    else:
        container.children = [tab]
    
    # Display the tabbed interface
    display(container)


In [5]:
def plot_shared_features_over_time(data, shared_features, filehash):
    """Plot a selection of shared features for both players over time."""
    match = data[data["filehash"] == filehash]
    feature_plots = list()

    for feature in shared_features:
        def plot_function(feature=feature):
            plt.figure(figsize=(10, 5))
            sns.lineplot(x="seconds", y=f"player_1_{feature}", color="blue",data=match, label="player_1")
            sns.lineplot(x="seconds", y=f"player_2_{feature}", color="orange", data=match, label="player_2")
            plt.title(f"{feature} over time")
            plt.xlabel("Time (seconds)")
            plt.ylabel(feature)
            plt.legend()
        feature_plots.append(plot_function)

    create_tabbed_plots(feature_plots, shared_features, overall_title=f"Winner: {match['winner'].unique()[0]}")

In [58]:
def plot_player_features_over_time(data, unit_features, filehash, player_number):
    """Plot individual player features over time, such as unit counts, etc."""
    match = data[data["filehash"] == filehash]
    feature_plots = list()

    for feature in unit_features:
        def plot_function(feature=feature):
            plt.figure(figsize=(10, 5))
            sns.lineplot(x="seconds", y=f"player_{player_number}_{feature}", color="blue",data=match, label=f"player_{player_number}")
            plt.title(f"{feature} over time")
            plt.xlabel("Time (seconds)")
            plt.ylabel(feature)
            plt.legend()
            plt.show()
        feature_plots.append(plot_function)

    create_tabbed_plots(feature_plots, unit_features, overall_title=f"Winner: {match['winner'].unique()[0]}")

### Load Data

In [32]:
data = {
    "Protoss vs Protoss": pd.read_pickle("../data/Protoss_vs_Protoss.pkl"),
    "Zerg vs Zerg": pd.read_pickle("../data/Zerg_vs_Zerg.pkl"),
    "Terran vs Terran": pd.read_pickle("../data/Terran_vs_Terran.pkl"),
    "Protoss vs Zerg": pd.read_pickle("../data/Protoss_vs_Zerg.pkl"),
    "Terran vs Zerg": pd.read_pickle("../data/Terran_vs_Zerg.pkl"),
    "Protoss vs Terran": pd.read_pickle("../data/Protoss_vs_Terran.pkl"),
}

## High Level Analysis
Look at some general statistics about each matchup, before diving into specific matchup features

In [33]:
for matchup, data_sample in data.items():
    print(matchup)
    print("number of games: ", data_sample["filehash"].nunique())
    print("number of features: ", data_sample.shape[1])
    print("average target: ", data_sample[data_sample["seconds"] == 0]["winner"].mean())
    print("\n")


Protoss vs Protoss
number of games:  250
number of features:  119
average target:  1.50199203187251


Zerg vs Zerg
number of games:  89
number of features:  131
average target:  1.5280898876404494


Terran vs Terran
number of games:  200
number of features:  133
average target:  1.47


Protoss vs Zerg
number of games:  346
number of features:  125
average target:  1.5635838150289016


Terran vs Zerg
number of games:  302
number of features:  132
average target:  1.4602649006622517


Protoss vs Terran
number of games:  453
number of features:  126
average target:  1.5805739514348787




All 'average target' values are ~1.5, which is what we would expect (the target right now is 1 or 2, not 0 or 1).

### Check for broken features
Here we check for broken features. We check for any features which:
- Never contain a non-zero count across all games

This is to ensure that the feature extraction process during the replay ingestion is not broken. We should expect to see all units being built at least once, and we should expect units to die throughout games.

In [34]:
# generate the full dataset, which is a concatination of all the sub datasets
full_data = pd.concat(list(data.values()), ignore_index=True)
full_data.fillna(0, inplace=True)

In [35]:
for race in ["Protoss", "Terran", "Zerg"]:
    for unit in TRACKED_UNIT_TYPES[race]:
        features = [x for x in [f"player_1_{unit}", f"player_2_{unit}"] if x in full_data.columns]
        for feature in features:
            if full_data[feature].max() == 0:
                print(feature)
        

player_1_Ultralisk
player_1_BroodLord


**Conclusion:** Only Ultralisk and BroodLord and missing for `player_1`. Due to our matchups being ordered alphabetically, `player_1` Zerg only happens in ZvZ mirror matchups. It is reasonable to assume that in all of our replays, no Ultralisks or BroodLords were made during ZvZ.

In [36]:
# generate all unit features in the full_data
all_unit_features = []
for race in ["Protoss", "Terran", "Zerg"]:
    for unit in TRACKED_UNIT_TYPES[race]:
        all_unit_features += [x for x in [f"player_1_{unit}", f"player_2_{unit}"] if x in full_data.columns]

In [37]:
# create a data subset with all features shifted by 1. This allows us to calcualte the change in feature.
# We should see some negative changes in features, meaning units are dying!
data_subset = full_data[["seconds"] + all_unit_features].copy()
for feature in all_unit_features:
    data_subset[feature] = data_subset[feature] - data_subset[feature].shift(1)

# remove seconds == 0 which will be the handover row between games where the shift logic makes no sense
data_subset = data_subset[data_subset["seconds"] == 0]

In [38]:
for feature in all_unit_features:
    if data_subset[feature].min() >= 0:
        print(feature)
        print(data_subset[feature].max())

player_1_SwarmHostMP
0.0
player_1_Ultralisk
0.0
player_1_Corruptor
0.0
player_1_BroodLord
0.0


Again, the only non-decreasing units are zerg units in ZvZ matchups. We can assume this is accurate and that our data loading is correct.

## Matchup Analysis

### Mirror Matchup

In [39]:
matchup = "Zerg vs Zerg"

In [40]:
filehash = data[matchup]["filehash"].unique()[0]
race = matchup.split(" ")[0]

In [41]:
# plot player stats fields over time
plot_shared_features_over_time(data[matchup], PLAYER_STATS_FIELDS, filehash)

In [43]:
# can plot upgrade features as shared features for mirror matchups
plot_shared_features_over_time(data[matchup], TRACKED_UNIT_TYPES[race], filehash)

In [42]:
# can plot upgrade features as shared features for mirror matchups
plot_shared_features_over_time(data[matchup], TRACKED_UPGRADE_TYPES[race], filehash)

### Non-mirror Matchup

In [48]:
matchup = "Terran vs Zerg"

In [49]:
filehash = data[matchup]["filehash"].unique()[0]
race_1 = matchup.split(" ")[0]
race_2 = matchup.split(" ")[-1]

In [59]:
# plot player stats fields over time
plot_shared_features_over_time(data[matchup], PLAYER_STATS_FIELDS, filehash)

In [60]:
# plot player 1 units over time
plot_player_features_over_time(data[matchup], TRACKED_UNIT_TYPES[race_1], filehash, 1)

In [61]:
# plot player 2 units over time
plot_player_features_over_time(data[matchup], TRACKED_UNIT_TYPES[race_2], filehash, 2)

In [62]:
# plot player 1 upgrades over time
plot_player_features_over_time(data[matchup], TRACKED_UPGRADE_TYPES[race_1], filehash, 1)

In [63]:
# plot player 2 upgrades over time
plot_player_features_over_time(data[matchup], TRACKED_UPGRADE_TYPES[race_2], filehash, 2)